# Lab 02: Basic NLP Preprocessing Techniques

**Course:** ITAI 2373 - Natural Language Processing  
**Module:** 02 - Text Preprocessing  
**Duration:** 2-3 hours  
**Student Name:** ________________  
**Date:** ________________

---

## 🎯 Learning Objectives

By completing this lab, you will:
1. Understand the critical role of preprocessing in NLP pipelines
2. Master fundamental text preprocessing techniques
3. Compare different libraries and their approaches
4. Analyze the effects of preprocessing on text data
5. Build a complete preprocessing pipeline
6. Load and work with different types of text datasets

## 📖 Introduction to NLP Preprocessing

Natural Language Processing (NLP) preprocessing refers to the initial steps taken to clean and transform raw text data into a format that's more suitable for analysis by machine learning algorithms.

### Why is preprocessing crucial?

1. **Standardization:** Ensures consistent text format across your dataset
2. **Noise Reduction:** Removes irrelevant information that could confuse algorithms
3. **Complexity Reduction:** Simplifies text to focus on meaningful patterns
4. **Performance Enhancement:** Improves the efficiency and accuracy of downstream tasks

### Real-world Impact
Consider searching for "running shoes" vs "Running Shoes!" - without preprocessing, these might be treated as completely different queries. Preprocessing ensures they're recognized as equivalent.

### 🤔 Conceptual Question 1
**Before we start coding, think about your daily interactions with text processing systems (search engines, chatbots, translation apps). What challenges do you think these systems face when processing human language? List at least 3 specific challenges and explain why each is problematic.**

*Double-click this cell to write your answer:*

**Challenge 1:**
**Ans: Lexical and Structural Ambiguity
Using words and sentences can have multiple valid interpretations that is depending on the context. There is lexical ambiguity occuring when a single word has multiple meanings. For example, "bank" as a financial institution vs. a riverbank, while structural ambiguity occurs when a  sentence's grammer allows multiple parses like "I saw the man with the telescope".

**Problematic - Without sufficient context, a system cannot determine the user's true intenet, leading to incorrect search results, mistranslations, or nonsensical chatbot responses.

**Challenge 2:**
**Ans: Polysemy, Idioms, and Non-Literal Language
Human communications relies heavily on sarcasm, metaphors, slang, and idiomatic expressions such as "spills the beans" or "kick the bucket".

**Problematic - Traditional or literal models interpret these phrases word-for-word, completing missing the intended meaning. Also detecting tone, humor, or figurative speech requires deep contextual awareness and cultural knowledge which has standard statistical models struggle to generalize.

**Challenge 3:**
**Ans: Anaphora and Coreference Resolution
In continuous text, pronouns and referring expressions such as like, he, she, it or the former, it is point back to previously mentioned entities.

**Problematic - In longer conversations  or multi-paragraph documents, tracking which entity a pronoun refers to across complex sentence structures is computationally difficult. A failure in coreference resolution leads to a loss of context and inaccuarte responses in multi-turn conversational systems.
---

## 🛠️ Part 1: Environment Setup

We'll be working with two major NLP libraries:
- **NLTK (Natural Language Toolkit):** Comprehensive NLP library with extensive resources
- **spaCy:** Industrial-strength NLP with pre-trained models

**⚠️ Note:** Installation might take 2-3 minutes to complete.

In [1]:
# Step 1: Install Required Libraries
print("🔧 Installing NLP libraries...")

!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

print("✅ Installation complete!")

🔧 Installing NLP libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 83.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Installation complete!


### 🤔 Conceptual Question 2
**Why do you think we need to install a separate language model (en_core_web_sm) for spaCy? What components might this model contain that help with text processing? Think about what information a computer needs to understand English text.**

*Double-click this cell to write your answer:*
**Ans:
We need to install a separate language model like en_core_web_sm for spaCy because spaCy is designed as a model-driven NLP engine, not just a set of text-processing algorithms. The reason for this is that while basic Python functions or simple libraries rely on hardcoded rule sets to process text, spaCy relies on pre-trained statistical and machine learning models to analyze the contextual rules and nuances of human language.

The componenets that this model might contain to help with the text processing are structural, grammatical, and semantic context. The en_core_web_sm model provides several integrated componenets which are Vocabulary & Vector Representations, Tokenizer Rules, Part-of-Speech Tagger, Dependency Parser, Named Entity Recognizer, and Lemmatizer.
Vocabulary & Vector Representations - is a lookup table and word embeddings that help the model recognize valid words, lowrercase variations and semantic relationships between words. Tokenizer Rules is a language-specific conventions that dictate how complex text (such as contractions like "don't" or punctuation attached to words) should be broken down into discrete tokens. Part-of-Speech  Tagger is a statistical weights that predict whether a word acts as a noun, verb, adjective, or adposition based on surrounding syntax. Dependency Parser is a neural network component that maps grammatical relationships within a sentence such as identifying subjects, objects, verb modifiers, and sentence boundaries. Named Entity Recognizer is a model trained to identify real-world entities inside text, such as proper names, organization names, dates, quantities, and locations. Lastly, lemmantizer is where morphological data tables used to map inflected word forms such as like "running", "ran", or "better" and back to their base dictionary forms such as "run", "good".

The reason it is dixtributed separately are its efficiency where keeping the language models separate from the core spacy package keeps the base library lightweight and allows users to download only the specific language and model size such as small, medium, or large that is required for their use case. Lastly, language specificity is where the grammar rules, word meanings, and syntax vary wildly across languages. Downloading en_core_web_sm explicitly configures spaCy to apply English-specific linguistic rules.

---

In [2]:
# Step 2: Import Libraries and Download NLTK Data
import nltk
import spacy
import string
import re
from collections import Counter

# Download essential NLTK data
print("📦 Downloading NLTK data packages...")
nltk.download('punkt')      # For tokenization
nltk.download('stopwords')  # For stop word removal
nltk.download('wordnet')    # For lemmatization
nltk.download('averaged_perceptron_tagger')  # For POS tagging

print("\n✅ All imports and downloads completed!")

📦 Downloading NLTK data packages...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...



✅ All imports and downloads completed!


[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


## 📂 Part 2: Sample Text Data

We'll work with different types of text to understand how preprocessing affects various text styles:
- Simple text
- Academic text (with citations, URLs)
- Social media text (with emojis, hashtags)
- News text (formal writing)
- Product reviews (informal, ratings)

In [3]:
# Step 3: Load Sample Texts
simple_text = "Natural Language Processing is a fascinating field of AI. It's amazing!"

academic_text = """
Dr. Smith's research on machine-learning algorithms is groundbreaking!
She published 3 papers in 2023, focusing on deep neural networks (DNNs).
The results were amazing - accuracy improved by 15.7%!
"This is revolutionary," said Prof. Johnson.
Visit https://example.com for more info. #NLP #AI @university
"""

social_text = "OMG! Just tried the new coffee shop ☕️ SO GOOD!!! Highly recommend 👍 #coffee #yum 😍"

news_text = """
The stock market experienced significant volatility today, with tech stocks leading the decline.
Apple Inc. (AAPL) dropped 3.2%, while Microsoft Corp. fell 2.8%.
"We're seeing a rotation out of growth stocks," said analyst Jane Doe from XYZ Capital.
"""

review_text = """
This laptop is absolutely fantastic! I've been using it for 6 months and it's still super fast.
The battery life is incredible - lasts 8-10 hours easily.
Only complaint: the keyboard could be better. Overall rating: 4.5/5 stars.
"""

# Store all texts
sample_texts = {
    "Simple": simple_text,
    "Academic": academic_text.strip(),
    "Social Media": social_text,
    "News": news_text.strip(),
    "Product Review": review_text.strip()
}

print("📄 Sample texts loaded successfully!")
for name, text in sample_texts.items():
    preview = text[:80] + "..." if len(text) > 80 else text
    print(f"\n🏷️ {name}: {preview}")

📄 Sample texts loaded successfully!

🏷️ Simple: Natural Language Processing is a fascinating field of AI. It's amazing!

🏷️ Academic: Dr. Smith's research on machine-learning algorithms is groundbreaking!
She publi...

🏷️ Social Media: OMG! Just tried the new coffee shop ☕️ SO GOOD!!! Highly recommend 👍 #coffee #yu...

🏷️ News: The stock market experienced significant volatility today, with tech stocks lead...

🏷️ Product Review: This laptop is absolutely fantastic! I've been using it for 6 months and it's st...


### 🤔 Conceptual Question 3
**Looking at the different text types we've loaded, what preprocessing challenges do you anticipate for each type? For each text type below, identify at least 2 specific preprocessing challenges and explain why they might be problematic for NLP analysis.**

*Double-click this cell to write your answer:*

**Simple text challenges:**
1. Contraction Ambiguity: Contractions like "It's" can split into "It" and "'s" or "is", which confuses basic word-boundary tokenizers and alters syntactic structures.
2. Loss of Subtle Context from Punctuation: Removing exclamations or quotes during basic cleaning can erase intensity or boundaries without providing a huge reduction in noise.

**Academic text challenges:**
1. In-text Citations and References: Citations contain names, years, and specific punctuation. For example "Smith et al., 2023" that can be mistakenly parsed as standard prose or fragmented into meaningless numerical tokens.
2. Domain-Specific Jargon and Abbreviations: Acronyms like "DNNs" or hyphenated terms like "machine-learning" can be incorrectly split by tokenizers or stripped by standard stop-word lists, losing critical semantic information.

**Social media text challenges:**
1. Emojis and Special Symbols: Emojis and non-standard characters carry heavy sentiment; stripping them during cleaning removes critical context needed for tasks like sentiment analysis.
2. Hashtags and User Mentions: Strings like #coffee or @university blend structural metadata with content; stripping the symbol can blur whether a word is an inline noun or an administrative tag.

**News text challenges:**
1.Named Entity Recognition Complexity: Frequent use of proper nouns, company symbols such as "Apple Inc, and titles like "Prof." and "Dr." can cause sentence tokenizers to prematurely split sentences at abbreviations.
2. Direct Quotations and Attribution: High density of quoted speech makes sentence boundary detection difficult and complicates tracking who said what if quotation marks are stripped.

**Product review challenges:**
1. Rating Scales and Numerical Data: Text containing ratings like "4.5/5 stars" or specs like "8-10 hours" can be ruined if digit-removal functions indiscriminately purge all numbers.
2. Informal Negations and Contrastive Language: Shift phrases and informal negations for example "Only complaint: ..." which is heavily rely on stop words like "not" or specific punctuation; aggressive stop-word or punctuation removal can flip the target sentiment.

---

## 🔤 Part 3: Tokenization

### What is Tokenization?
Tokenization is the process of breaking down text into smaller, meaningful units called **tokens**. These tokens are typically words, but can also be sentences, characters, or subwords.

### Why is it Important?
- Most NLP algorithms work with individual tokens, not entire texts
- It's the foundation for all subsequent preprocessing steps
- Different tokenization strategies can significantly impact results

### Common Challenges:
- **Contractions:** "don't" → "do" + "n't" or "don't"?
- **Punctuation:** Keep with words or separate?
- **Special characters:** How to handle @, #, URLs?

In [4]:
# Step 4: Tokenization with NLTK
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk
nltk.download('punkt_tab')

# Test on simple text
print("🔍 NLTK Tokenization Results")
print("=" * 40)
print(f"Original: {simple_text}")

# Word tokenization
nltk_tokens = word_tokenize(simple_text)
print(f"\nWord tokens: {nltk_tokens}")
print(f"Number of tokens: {len(nltk_tokens)}")

# Sentence tokenization
sentences = sent_tokenize(simple_text)
print(f"\nSentences: {sentences}")
print(f"Number of sentences: {len(sentences)}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


🔍 NLTK Tokenization Results
Original: Natural Language Processing is a fascinating field of AI. It's amazing!

Word tokens: ['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', 'of', 'AI', '.', 'It', "'s", 'amazing', '!']
Number of tokens: 14

Sentences: ['Natural Language Processing is a fascinating field of AI.', "It's amazing!"]
Number of sentences: 2


### 🤔 Conceptual Question 4
**Examine the NLTK tokenization results above. How did NLTK handle the contraction "It's"? What happened to the punctuation marks? Do you think this approach is appropriate for all NLP tasks? Explain your reasoning.**

*Double-click this cell to write your answer:*

**How "It's" was handled:**
**Ans:NLTK splits the contraction into two separate tokens such as It and 's. It separates the base pronoun from the verb component while keeping the apostrophe attached to the suffix.

**Punctuation treatment:**
**Ans: NLTK isolates punctuation marks into individual, independent tokens. For example, the exclamation mark in "amazing!" is separated into its own token rather than remaining attached to the word.

**Appropriateness for different tasks:**
**Ans: No, this approach is not appropriate for all NLP tasks:
Suitable Tasks: It works well for syntactic parsing, Part-of-Speech tagging , and language modeling where separating contractions into distinct grammatical components such as pronoun + verb that helps models learn grammatical structures.
Unsuitable Tasks: It can be detrimental for tasks like social media analysis, sentiment analysis, or search indexing where separating punctuation or splitting contractions might discard stylistic cues, alter sentiment context such as "don't" vs. "do" + "n't"), or break token-to-word mappings.

---

In [5]:
# Step 5: Tokenization with spaCy
nlp = spacy.load('en_core_web_sm')

print("🔍 spaCy Tokenization Results")
print("=" * 40)
print(f"Original: {simple_text}")

# Process with spaCy
doc = nlp(simple_text)

# Extract tokens
spacy_tokens = [token.text for token in doc]
print(f"\nWord tokens: {spacy_tokens}")
print(f"Number of tokens: {len(spacy_tokens)}")

# Show detailed token information
print(f"\n🔬 Detailed Token Analysis:")
print(f"{'Token':<12} {'POS':<8} {'Lemma':<12} {'Is Alpha':<8} {'Is Stop':<8}")
print("-" * 50)
for token in doc:
    print(f"{token.text:<12} {token.pos_:<8} {token.lemma_:<12} {token.is_alpha:<8} {token.is_stop:<8}")

🔍 spaCy Tokenization Results
Original: Natural Language Processing is a fascinating field of AI. It's amazing!

Word tokens: ['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', 'of', 'AI', '.', 'It', "'s", 'amazing', '!']
Number of tokens: 14

🔬 Detailed Token Analysis:
Token        POS      Lemma        Is Alpha Is Stop 
--------------------------------------------------
Natural      PROPN    Natural      1        0       
Language     PROPN    Language     1        0       
Processing   NOUN     processing   1        0       
is           AUX      be           1        1       
a            DET      a            1        1       
fascinating  ADJ      fascinating  1        0       
field        NOUN     field        1        0       
of           ADP      of           1        1       
AI           PROPN    AI           1        0       
.            PUNCT    .            0        0       
It           PRON     it           1        1       
's           AUX     

### 🤔 Conceptual Question 5
**Compare the NLTK and spaCy tokenization results. What differences do you notice? Which approach do you think would be better for different NLP tasks? Consider specific examples like sentiment analysis vs. information extraction.**

*Double-click this cell to write your answer:*

**Key differences observed:**
**Ans:
Contraction Handling is where NLTK splits contractions like "It's" into "It" and "'s", whereas spaCy processes tokenization dependency pipelines using fixed rule-based models alongside deep linguistic annotations
Punctuation & Features is where NLTK treats punctuation as individual string tokens. spaCy keeps punctuation as tokens while attaching explicit attributes to each token, such as part-of-speech tags,  lemmas, and boolean flags.  
Architecture is where NLTK relies on isolated algorithms for different tasks. For instance,  separate tokenizers, stemmers, taggers. SpaCy processes text in an integrated pipeline using statistical and transformer models.

**Better for sentiment analysis:**
**Ans:
spaCy/taileored NLTK rule sets
Sentiment analysis requires preserving negation and emotional modifiers. For example "not", "n't", emojis, and punctuation like "!". spaCy handles complex real-world text, emojis, and multi-token contractions consistently without stripping away structural meaning


**Better for information extraction:**
**Ans:
spaCy
The information extraction relies heavily on underlying linguistic context, dependency parsing part-of-speech tagging, and named entity recognition. spaCy provides detailed token-level attributes out of the box, allowing models to extract relationships and entities accurately.

**Overall assessment:**
**Ans:
NLTK provides a customizable toolkit for granular algorithmic control and lightweight string manipulation, making it ideal for academic experimentation. spaCy provides an end-to-end, object-oriented pipeline optimized for production-level NLP tasks that require rich contextual analysis and fast execution.

---

In [6]:
# Step 6: Test Tokenization on Complex Text
print("🧪 Testing on Social Media Text")
print("=" * 40)
print(f"Original: {social_text}")

# NLTK approach
social_nltk_tokens = word_tokenize(social_text)
print(f"\nNLTK tokens: {social_nltk_tokens}")

# spaCy approach
social_doc = nlp(social_text)
social_spacy_tokens = [token.text for token in social_doc]
print(f"spaCy tokens: {social_spacy_tokens}")

print(f"\n📊 Comparison:")
print(f"NLTK token count: {len(social_nltk_tokens)}")
print(f"spaCy token count: {len(social_spacy_tokens)}")

🧪 Testing on Social Media Text
Original: OMG! Just tried the new coffee shop ☕️ SO GOOD!!! Highly recommend 👍 #coffee #yum 😍

NLTK tokens: ['OMG', '!', 'Just', 'tried', 'the', 'new', 'coffee', 'shop', '☕️', 'SO', 'GOOD', '!', '!', '!', 'Highly', 'recommend', '👍', '#', 'coffee', '#', 'yum', '😍']
spaCy tokens: ['OMG', '!', 'Just', 'tried', 'the', 'new', 'coffee', 'shop', '☕', '️', 'SO', 'GOOD', '!', '!', '!', 'Highly', 'recommend', '👍', '#', 'coffee', '#', 'yum', '😍']

📊 Comparison:
NLTK token count: 22
spaCy token count: 23


### 🤔 Conceptual Question 6
**Looking at how the libraries handled social media text (emojis, hashtags), which library seems more robust for handling "messy" real-world text? What specific advantages do you notice? How might this impact a real-world application like social media sentiment analysis?**

*Double-click this cell to write your answer:*

**More robust library:**
**Ans: spaCy

**Specific advantages:**
**Ans:
Keeps emojis intact as distinct, individual tokens that can separte unts rather then spilitting or mangling unicode byte sequences.
Isolates the hashtag symbol cleanly while keeping hashtag text intact, rather than breking multi-word hashtags unpredictably.
Hnadles punctuation clustering such as repeated exclamation marks and contractions cleanly without over-segmenting social media slang.

**Impact on sentiment analysis:**
**Ans:
Preserves Strong Emotional Signals is where the emojis are primary indicators of sentiment in modern text. Stripping or splitting them degrades classifier accuracy.
Retains Topic & Intent Markers is where hashtags often carry the core context or intent of a post. Isolating them reliably allows sentiment models to attribute mood to specific topics
Improves Feature Vector Quality is where cleaner tokenization reduces overall vocabulary noise, leading to higher accuracy and fewer out-of-vocabulary errors when feeding text into downstream machine learning models or sentiment dictionaries.

---

## 🛑 Part 4: Stop Words Removal

### What are Stop Words?
Stop words are common words that appear frequently in a language but typically don't carry much meaningful information about the content. Examples include "the", "is", "at", "which", "on", etc.

### Why Remove Stop Words?
1. **Reduce noise** in the data
2. **Improve efficiency** by reducing vocabulary size
3. **Focus on content words** that carry semantic meaning

### When NOT to Remove Stop Words?
- **Sentiment analysis:** "not good" vs "good" - the "not" is crucial!
- **Question answering:** "What is the capital?" - "what" and "is" provide context

In [7]:
# Step 7: Explore Stop Words Lists
from nltk.corpus import stopwords

# Get NLTK English stop words
nltk_stopwords = set(stopwords.words('english'))
print(f"📊 NLTK has {len(nltk_stopwords)} English stop words")
print(f"First 20: {sorted(list(nltk_stopwords))[:20]}")

# Get spaCy stop words
spacy_stopwords = nlp.Defaults.stop_words
print(f"\n📊 spaCy has {len(spacy_stopwords)} English stop words")
print(f"First 20: {sorted(list(spacy_stopwords))[:20]}")

# Compare the lists
common_stopwords = nltk_stopwords.intersection(spacy_stopwords)
nltk_only = nltk_stopwords - spacy_stopwords
spacy_only = spacy_stopwords - nltk_stopwords

print(f"\n🔍 Comparison:")
print(f"Common stop words: {len(common_stopwords)}")
print(f"Only in NLTK: {len(nltk_only)} - Examples: {sorted(list(nltk_only))[:5]}")
print(f"Only in spaCy: {len(spacy_only)} - Examples: {sorted(list(spacy_only))[:5]}")

📊 NLTK has 198 English stop words
First 20: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

📊 spaCy has 326 English stop words
First 20: ["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also']

🔍 Comparison:
Common stop words: 123
Only in NLTK: 75 - Examples: ['ain', 'aren', "aren't", 'couldn', "couldn't"]
Only in spaCy: 203 - Examples: ["'d", "'ll", "'m", "'re", "'s"]


### 🤔 Conceptual Question 7
**Why do you think NLTK and spaCy have different stop word lists? Look at the examples of words that are only in one list - do you agree with these choices? Can you think of scenarios where these differences might significantly impact your NLP results?**

*Double-click this cell to write your answer:*

**Reasons for differences:**
**Ans:
NLTK and spaCy were designed for different purposes and built at different times. NLTK is an academic toolkit that uses traditional, curational stop word lists designed primarily for classic document indexing and frequency-based modeling. spaCy, designed as an industrial-strength library, uses a larger, modern standard set that includes a wider variety of functional words (such as pronoums, adverbs, and multi-word contraction components) to maximize vocabulary reduction for modern machine learning pipelines.

**Agreement with choices:**
**Ans:
Agree: I agree with spaCy's decision to include words related to modern conversational English, pronouns, and quantity descriptors such as like "much", "many", "several", as these rarely carry specific domain content in classic topic modeling or document classification tasks.

Disagree: I disagree with aggressive stop word lists that remove negation words such as "not", "nor", "never" or modal verbs. SpaCy and NLTK both include some numbers or directional terms that can be vital depending on context. Removing negations strips away essential meaning for tasks that rely on semantic polarity.

**Scenarios where differences matter:**
**Ans:
Sentiment Analysis is where spaCy includes modal verbs, contractions, and adverbs that drastically alter polarity. If a stop word list removes "not" or "never", a sentiment model might treat "not good" as identical to "good".
Question Answering & Search Information Retrieval is where queries like "to be or not to be" or "where am I" consist almost entirely of stop words. NLTK and spaCy handle pronouns and auxiliary verbs differently, which can result in completely dropping critical query tokens in one library versus the other.
Text Summarization & Keyphrase Extraction is when NLTK’s smaller list might leave behind transitional or functional words that contaminate keyphrase rankings, whereas spaCy broader list might accidentally eliminate context-defining connector words.


---

In [8]:
# Step 8: Remove Stop Words with NLTK
# Test on simple text
original_tokens = nltk_tokens  # From earlier tokenization
filtered_tokens = [word for word in original_tokens if word.lower() not in nltk_stopwords]

print("🧪 NLTK Stop Word Removal")
print("=" * 40)
print(f"Original: {simple_text}")
print(f"\nOriginal tokens ({len(original_tokens)}): {original_tokens}")
print(f"After removing stop words ({len(filtered_tokens)}): {filtered_tokens}")

# Show which words were removed
removed_words = [word for word in original_tokens if word.lower() in nltk_stopwords]
print(f"\nRemoved words: {removed_words}")

# Calculate reduction percentage
reduction = (len(original_tokens) - len(filtered_tokens)) / len(original_tokens) * 100
print(f"Vocabulary reduction: {reduction:.1f}%")

🧪 NLTK Stop Word Removal
Original: Natural Language Processing is a fascinating field of AI. It's amazing!

Original tokens (14): ['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', 'of', 'AI', '.', 'It', "'s", 'amazing', '!']
After removing stop words (10): ['Natural', 'Language', 'Processing', 'fascinating', 'field', 'AI', '.', "'s", 'amazing', '!']

Removed words: ['is', 'a', 'of', 'It']
Vocabulary reduction: 28.6%


In [9]:
# Step 9: Remove Stop Words with spaCy
doc = nlp(simple_text)
spacy_filtered = [token.text for token in doc if not token.is_stop and not token.is_punct]

print("🧪 spaCy Stop Word Removal")
print("=" * 40)
print(f"Original: {simple_text}")
print(f"\nOriginal tokens ({len(spacy_tokens)}): {spacy_tokens}")
print(f"After removing stop words & punctuation ({len(spacy_filtered)}): {spacy_filtered}")

# Show which words were removed
spacy_removed = [token.text for token in doc if token.is_stop or token.is_punct]
print(f"\nRemoved words: {spacy_removed}")

# Calculate reduction percentage
spacy_reduction = (len(spacy_tokens) - len(spacy_filtered)) / len(spacy_tokens) * 100
print(f"Vocabulary reduction: {spacy_reduction:.1f}%")

🧪 spaCy Stop Word Removal
Original: Natural Language Processing is a fascinating field of AI. It's amazing!

Original tokens (14): ['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', 'of', 'AI', '.', 'It', "'s", 'amazing', '!']
After removing stop words & punctuation (7): ['Natural', 'Language', 'Processing', 'fascinating', 'field', 'AI', 'amazing']

Removed words: ['is', 'a', 'of', '.', 'It', "'s", '!']
Vocabulary reduction: 50.0%


### 🤔 Conceptual Question 8
**Compare the NLTK and spaCy stop word removal results. Which approach removed more words? Do you think removing punctuation (as spaCy did) is always a good idea? Give a specific example where keeping punctuation might be important for NLP analysis.**

*Double-click this cell to write your answer:*

**Which removed more:**
**Ans: spaCy removed more items because its pipeline explicity filtered out both stop words and punctuation. Additionally, spaCy utilizes a larger default stop word dictionary compared to NLTK.

**Punctuation removal assessment:**
**Ans: No, emoving punctuation is not always a good idea. Punctuation carries crucial syntactic, sentiment, and structural information that helps downstream NLP models interpret context, tone, and intent.

**Example where punctuation matters:**
**Ans: In sentiment analysis and intent detection, punctuation alters meaning drastically
Example: "This product is bad!" vs. "This product is bad?"
Exclamation marks signal strong negative sentiment, whereas a question mark changes the statement to a query or doubt. Removing punctuation strips away these critical tonal nuances.
---

## 🌱 Part 5: Lemmatization and Stemming

### What is Lemmatization?
Lemmatization reduces words to their base or dictionary form (called a **lemma**). It considers context and part of speech to ensure the result is a valid word.

### What is Stemming?
Stemming reduces words to their root form by removing suffixes. It's faster but less accurate than lemmatization.

### Key Differences:
| Aspect | Stemming | Lemmatization |
|--------|----------|---------------|
| Speed | Fast | Slower |
| Accuracy | Lower | Higher |
| Output | May be non-words | Always valid words |
| Context | Ignores context | Considers context |

### Examples:
- **"running"** → Stem: "run", Lemma: "run"
- **"better"** → Stem: "better", Lemma: "good"
- **"was"** → Stem: "wa", Lemma: "be"

In [10]:
# Step 10: Stemming with NLTK
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

# Test words that demonstrate stemming challenges
test_words = ['running', 'runs', 'ran', 'better', 'good', 'best', 'flying', 'flies', 'was', 'were', 'cats', 'dogs']

print("🌿 Stemming Demonstration")
print("=" * 30)
print(f"{'Original':<12} {'Stemmed':<12}")
print("-" * 25)

for word in test_words:
    stemmed = stemmer.stem(word)
    print(f"{word:<12} {stemmed:<12}")

# Apply to our sample text
sample_tokens = [token for token in nltk_tokens if token.isalpha()]
stemmed_tokens = [stemmer.stem(token.lower()) for token in sample_tokens]

print(f"\n🧪 Applied to sample text:")
print(f"Original: {sample_tokens}")
print(f"Stemmed: {stemmed_tokens}")

🌿 Stemming Demonstration
Original     Stemmed     
-------------------------
running      run         
runs         run         
ran          ran         
better       better      
good         good        
best         best        
flying       fli         
flies        fli         
was          wa          
were         were        
cats         cat         
dogs         dog         

🧪 Applied to sample text:
Original: ['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', 'of', 'AI', 'It', 'amazing']
Stemmed: ['natur', 'languag', 'process', 'is', 'a', 'fascin', 'field', 'of', 'ai', 'it', 'amaz']


### 🤔 Conceptual Question 9
**Look at the stemming results above. Can you identify any cases where stemming produced questionable results? For example, how were "better" and "good" handled? Do you think this is problematic for NLP applications? Explain your reasoning.**

*Double-click this cell to write your answer:*

**Questionable results identified:**
**Ans:
"was" to "wa" and "flies" to "fli" - the Porter  Stemmer blindly chops off standard suffixes like -s or -es producing non-words or truncated roots rather than meaningful base terms.
"better" to "better" is where the irregular forms are ignored by simple heuristic rules and leaving the word completely unchanged.
Loss of distinct semantic meanings of words such as "flies" lose their grammatical structure without resolving to a proper dictionary word.

**Assessment of "better" and "good":**
**Ans:
Porter Stemming evaluates words based purely on surface-level, rule-based suffix removal.
Both "better" and "good" are irregular forms (suppletion) of the same root adjective, rule-based stemming fails to map "better" to "good" or its root form.
Stemming treats "better" and "good" as completely unrelated tokens.

**Impact on NLP applications:**
**Ans:
Problematic for Semantic Understanding & Search is where the downstream models treat "better" and "good" as distinct concepts rather than related sentiments or synonyms, causing missed matches in search queries and informational retrieval.
Loss of Context in Sentiment Analysis is where truncated non-words or failing to collapse irregular comparative forms hinders sentiment scores and vocabulary standardization.
When stemming is acceptable is where the remains are useful for high-speed, lightweight tasks where fast keyword matching and reduced memory usage outweigh semantic accuracy. However, for context-dependent NLP, lemmatization or contextual embeddings are preferred.

---

In [11]:
# Step 11: Lemmatization with spaCy
print("🌱 spaCy Lemmatization Demonstration")
print("=" * 40)

# Test on a complex sentence
complex_sentence = "The researchers were studying the effects of running and swimming on better performance."
doc = nlp(complex_sentence)

print(f"Original: {complex_sentence}")
print(f"\n{'Token':<15} {'Lemma':<15} {'POS':<10} {'Explanation':<20}")
print("-" * 65)

for token in doc:
    if token.is_alpha:
        explanation = "No change" if token.text.lower() == token.lemma_ else "Lemmatized"
        print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<10} {explanation:<20}")

# Extract lemmas
lemmas = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop]
print(f"\n🔤 Lemmatized tokens (no stop words): {lemmas}")

🌱 spaCy Lemmatization Demonstration
Original: The researchers were studying the effects of running and swimming on better performance.

Token           Lemma           POS        Explanation         
-----------------------------------------------------------------
The             the             DET        No change           
researchers     researcher      NOUN       Lemmatized          
were            be              AUX        Lemmatized          
studying        study           VERB       Lemmatized          
the             the             DET        No change           
effects         effect          NOUN       Lemmatized          
of              of              ADP        No change           
running         run             VERB       Lemmatized          
and             and             CCONJ      No change           
swimming        swim            VERB       Lemmatized          
on              on              ADP        No change           
better          well          

In [12]:
# Step 12: Compare Stemming vs Lemmatization
comparison_words = ['better', 'running', 'studies', 'was', 'children', 'feet']

print("⚖️ Stemming vs Lemmatization Comparison")
print("=" * 50)
print(f"{'Original':<12} {'Stemmed':<12} {'Lemmatized':<12}")
print("-" * 40)

for word in comparison_words:
    # Stemming
    stemmed = stemmer.stem(word)

    # Lemmatization with spaCy
    doc = nlp(word)
    lemmatized = doc[0].lemma_

    print(f"{word:<12} {stemmed:<12} {lemmatized:<12}")

⚖️ Stemming vs Lemmatization Comparison
Original     Stemmed      Lemmatized  
----------------------------------------
better       better       well        
running      run          run         
studies      studi        study       
was          wa           be          
children     children     child       
feet         feet         foot        


### 🤔 Conceptual Question 10
**Compare the stemming and lemmatization results. Which approach do you think is more suitable for:**
1. **A search engine** (where speed is crucial and you need to match variations of words)?
2. **A sentiment analysis system** (where accuracy and meaning preservation are important)?
3. **A real-time chatbot** (where both speed and accuracy matter)?

**Explain your reasoning for each choice.**

*Double-click this cell to write your answer:*

**1. Search engine:**
**Ans: Stemming is where search engines prioritize rapid retrieval speeds and maximum recall across large datasets. Stemming relies on simple heuristic rules to strip suffixes without dictionary lookups, allowing it to execute rapidly while grouping words variants under the same stem to broaden search query matches.

**2. Sentiment analysis:**
**Ans: Lemmatization is where sentiment analysis relies on high accuracy and strict preservation of context and semantic meaning. Lemmatization utilizes part-of-speeh context and valid dictionary lemmas, ensuring nuance, negation, and emotional intensity are properly analyzed without outputting invalid or distorted roots.

**3. Real-time chatbot:**
**Ans: Lemmatization/A modern hybrid approach is where chatbots require precise natural language understanding to maintain coherent, context-aware conversations while operating with low latency. Lemmatization ensures intent and meaning are preserved without producing non-words, and modern NLP frameworks process lemmatization fast enough to handle real-time user interactions effectively without compromising response accuracy.

---

## 🧹 Part 6: Text Cleaning and Normalization

### What is Text Cleaning?
Text cleaning involves removing or standardizing elements that might interfere with analysis:
- **Case normalization** (converting to lowercase)
- **Punctuation removal**
- **Number handling** (remove, replace, or normalize)
- **Special character handling** (URLs, emails, mentions)
- **Whitespace normalization**

### Why is it Important?
- Ensures consistency across your dataset
- Reduces vocabulary size
- Improves model performance
- Handles edge cases in real-world data

In [13]:
# Step 13: Basic Text Cleaning
def basic_clean_text(text):
    """Apply basic text cleaning operations"""
    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces again
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Test basic cleaning
test_text = "   Hello WORLD!!! This has 123 numbers and   extra spaces.   "
cleaned = basic_clean_text(test_text)

print("🧹 Basic Text Cleaning")
print("=" * 30)
print(f"Original: '{test_text}'")
print(f"Cleaned: '{cleaned}'")
print(f"Length reduction: {(len(test_text) - len(cleaned))/len(test_text)*100:.1f}%")

🧹 Basic Text Cleaning
Original: '   Hello WORLD!!! This has 123 numbers and   extra spaces.   '
Cleaned: 'hello world this has numbers and extra spaces'
Length reduction: 26.2%


In [14]:
# Step 14: Advanced Cleaning for Social Media
def advanced_clean_text(text):
    """Apply advanced cleaning for social media and web text"""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)

    # Convert hashtags (keep the word, remove #)
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove emojis (basic approach)
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

    # Convert to lowercase and normalize whitespace
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Test on social media text
print("🚀 Advanced Cleaning on Social Media Text")
print("=" * 45)
print(f"Original: {social_text}")

cleaned_social = advanced_clean_text(social_text)
print(f"Cleaned: {cleaned_social}")
print(f"Length reduction: {(len(social_text) - len(cleaned_social))/len(social_text)*100:.1f}%")

🚀 Advanced Cleaning on Social Media Text
Original: OMG! Just tried the new coffee shop ☕️ SO GOOD!!! Highly recommend 👍 #coffee #yum 😍
Cleaned: omg! just tried the new coffee shop ☕️ so good!!! highly recommend coffee yum
Length reduction: 7.2%


### 🤔 Conceptual Question 11
**Look at the advanced cleaning results for the social media text. What information was lost during cleaning? Can you think of scenarios where removing emojis and hashtags might actually hurt your NLP application? What about scenarios where keeping them would be beneficial?**

*Double-click this cell to write your answer:*

**Information lost:**
**Ans:
Emotional Nuance & Tone where the emoji's are stripped out which removes explicit emotional markers and intensity.
Intent & Context signals is where the hashtag symbols were stripped, leaving raw keywords such as coffee and yum being detached from their catwgorization function as social media metadata.
Stylistic Emphasis is where multiple punctuation marks, capitalization and non-standard spacing were flattened or removed.

**Scenarios where removal hurts:**
**Ans:
Sentiment Analysis & Emotion Detection is deleting emojis strips vital signals that models rely on to guage sentiment correctly.
Topic Classification & Brand Monitoring is removing hashtags degrade the ability to group posts by explicit user-defined metadata or track trending campaigns.
Aspect-Based Sentiment is where the emojis often clarify which part of a text carries sentiment aiding fine-grained target recognition

**Scenarios where keeping helps:**
**Ans:
Training Social Media Embeddings is where the emojis and hashtaags carry rich semantic weight, preserving them improves representation learning for platforms like twitter or Instagram.
Sarcasm & Irony Detection is emojis that are often signal that the literal text means the exact opposite of what is written.
Low-Resource & Slang Processing where hashtags frequently capture emerging slang, event tags or multi-word concepts where standard vocabulary lists miss.
---

## 🔧 Part 7: Building a Complete Preprocessing Pipeline

Now let's combine everything into a comprehensive preprocessing pipeline that you can customize based on your needs.

### Pipeline Components:
1. **Text cleaning** (basic or advanced)
2. **Tokenization** (NLTK or spaCy)
3. **Stop word removal** (optional)
4. **Lemmatization/Stemming** (optional)
5. **Additional filtering** (length, etc.)

In [15]:
# Step 15: Complete Preprocessing Pipeline
def preprocess_text(text,
                   clean_level='basic',     # 'basic' or 'advanced'
                   remove_stopwords=True,
                   use_lemmatization=True,
                   use_stemming=False,
                   min_length=2):
    """
    Complete text preprocessing pipeline
    """
    # Step 1: Clean text
    if clean_level == 'basic':
        cleaned_text = basic_clean_text(text)
    else:
        cleaned_text = advanced_clean_text(text)

    # Step 2: Tokenize
    if use_lemmatization:
        # Use spaCy for lemmatization
        doc = nlp(cleaned_text)
        tokens = [token.lemma_.lower() for token in doc if token.is_alpha]
    else:
        # Use NLTK for basic tokenization
        tokens = word_tokenize(cleaned_text)
        tokens = [token for token in tokens if token.isalpha()]

    # Step 3: Remove stop words
    if remove_stopwords:
        if use_lemmatization:
            tokens = [token for token in tokens if token not in spacy_stopwords]
        else:
            tokens = [token.lower() for token in tokens if token.lower() not in nltk_stopwords]

    # Step 4: Apply stemming if requested
    if use_stemming and not use_lemmatization:
        tokens = [stemmer.stem(token.lower()) for token in tokens]

    # Step 5: Filter by length
    tokens = [token for token in tokens if len(token) >= min_length]

    return tokens

print("🔧 Preprocessing Pipeline Created!")
print("✅ Ready to test different configurations.")

🔧 Preprocessing Pipeline Created!
✅ Ready to test different configurations.


In [16]:
# Step 16: Test Different Pipeline Configurations
test_text = sample_texts["Product Review"]
print(f"🎯 Testing on: {test_text[:100]}...")
print("=" * 60)

# Configuration 1: Minimal processing
minimal = preprocess_text(test_text,
                         clean_level='basic',
                         remove_stopwords=False,
                         use_lemmatization=False,
                         use_stemming=False)
print(f"\n1. Minimal processing ({len(minimal)} tokens):")
print(f"   {minimal[:10]}...")

# Configuration 2: Standard processing
standard = preprocess_text(test_text,
                          clean_level='basic',
                          remove_stopwords=True,
                          use_lemmatization=True)
print(f"\n2. Standard processing ({len(standard)} tokens):")
print(f"   {standard[:10]}...")

# Configuration 3: Aggressive processing
aggressive = preprocess_text(test_text,
                            clean_level='advanced',
                            remove_stopwords=True,
                            use_lemmatization=False,
                            use_stemming=True,
                            min_length=3)
print(f"\n3. Aggressive processing ({len(aggressive)} tokens):")
print(f"   {aggressive[:10]}...")

# Show reduction percentages
original_count = len(word_tokenize(test_text))
print(f"\n📊 Token Reduction Summary:")
print(f"   Original: {original_count} tokens")
print(f"   Minimal: {len(minimal)} ({(original_count-len(minimal))/original_count*100:.1f}% reduction)")
print(f"   Standard: {len(standard)} ({(original_count-len(standard))/original_count*100:.1f}% reduction)")
print(f"   Aggressive: {len(aggressive)} ({(original_count-len(aggressive))/original_count*100:.1f}% reduction)")

🎯 Testing on: This laptop is absolutely fantastic! I've been using it for 6 months and it's still super fast.
The ...

1. Minimal processing (34 tokens):
   ['this', 'laptop', 'is', 'absolutely', 'fantastic', 'ive', 'been', 'using', 'it', 'for']...

2. Standard processing (18 tokens):
   ['laptop', 'absolutely', 'fantastic', 've', 'use', 'month', 'super', 'fast', 'battery', 'life']...

3. Aggressive processing (21 tokens):
   ['laptop', 'absolut', 'fantast', 'use', 'month', 'still', 'super', 'fast', 'batteri', 'life']...

📊 Token Reduction Summary:
   Original: 47 tokens
   Minimal: 34 (27.7% reduction)
   Standard: 18 (61.7% reduction)
   Aggressive: 21 (55.3% reduction)


### 🤔 Conceptual Question 12
**Compare the three pipeline configurations (Minimal, Standard, Aggressive). For each configuration, analyze:**
1. **What information was preserved?**
2. **What information was lost?**
3. **What type of NLP task would this configuration be best suited for?**

*Double-click this cell to write your answer:*

**Minimal Processing:**
- Preserved:
**Ans: Full word tokens, contractions, original casing, numbers, and basic sentence structure
- Lost:
**Ans: Extra whitespace and punctuation marks.
- Best for:
**Ans: Complex sequence-dependent tasks like Neural Machine Translation (NMT), Language Modeling, or Sentiment Analysis where structural syntax and subtle cues matter.

**Standard Processing:**
- Preserved:
**Ans: Lemmatized word roots and core semantic content.
- Lost:
**Ans: Stop words, punctuation, original word inflection forms (verbs/nouns normalized), and specific tense details.
- Best for:
**Ans: Topic Modeling, Text Classification, and Information Retrieval (Search Engine Query Matching).

**Aggressive Processing:**
- Preserved:
**Ans: Truncated stemmed roots of non-stop words meeting the length threshold.
- Lost:
**Ans: Short words which has to be less than 3 characters, stop words, full dictionary word definitions/lemmas, special characters, and structural context.
- Best for:
**Ans: High-speed Bag-of-Words indexing, Spam Detection, or large-scale Document Clustering where aggressive dimensionality/vocabulary reduction is required.

---

In [17]:
# Step 17: Comprehensive Analysis Across Text Types
print("🔬 Comprehensive Preprocessing Analysis")
print("=" * 50)

# Test standard preprocessing on all text types
results = {}
for name, text in sample_texts.items():
    original_tokens = len(word_tokenize(text))
    processed_tokens = preprocess_text(text,
                                      clean_level='basic',
                                      remove_stopwords=True,
                                      use_lemmatization=True)

    reduction = (original_tokens - len(processed_tokens)) / original_tokens * 100
    results[name] = {
        'original': original_tokens,
        'processed': len(processed_tokens),
        'reduction': reduction,
        'sample': processed_tokens[:8]
    }

    print(f"\n📄 {name}:")
    print(f"   Original: {original_tokens} tokens")
    print(f"   Processed: {len(processed_tokens)} tokens ({reduction:.1f}% reduction)")
    print(f"   Sample: {processed_tokens[:8]}")

# Summary table
print(f"\n\n📋 Summary Table")
print(f"{'Text Type':<15} {'Original':<10} {'Processed':<10} {'Reduction':<10}")
print("-" * 50)
for name, data in results.items():
    print(f"{name:<15} {data['original']:<10} {data['processed']:<10} {data['reduction']:<10.1f}%")

🔬 Comprehensive Preprocessing Analysis

📄 Simple:
   Original: 14 tokens
   Processed: 7 tokens (50.0% reduction)
   Sample: ['natural', 'language', 'processing', 'fascinating', 'field', 'ai', 'amazing']

📄 Academic:
   Original: 61 tokens
   Processed: 26 tokens (57.4% reduction)
   Sample: ['dr', 'smith', 'research', 'machinelearning', 'algorithm', 'groundbreake', 'publish', 'paper']

📄 Social Media:
   Original: 22 tokens
   Processed: 10 tokens (54.5% reduction)
   Sample: ['omg', 'try', 'new', 'coffee', 'shop', 'good', 'highly', 'recommend']

📄 News:
   Original: 51 tokens
   Processed: 25 tokens (51.0% reduction)
   Sample: ['stock', 'market', 'experience', 'significant', 'volatility', 'today', 'tech', 'stock']

📄 Product Review:
   Original: 47 tokens
   Processed: 18 tokens (61.7% reduction)
   Sample: ['laptop', 'absolutely', 'fantastic', 've', 'use', 'month', 'super', 'fast']


📋 Summary Table
Text Type       Original   Processed  Reduction 
----------------------------------

### 🤔 Final Conceptual Question 13
**Looking at the comprehensive analysis results across all text types:**

1. **Which text type was most affected by preprocessing?** Why do you think this happened?

2. **Which text type was least affected?** What does this tell you about the nature of that text?

3. **If you were building an NLP system to analyze customer reviews for a business, which preprocessing approach would you choose and why?**

4. **What are the main trade-offs you need to consider when choosing preprocessing techniques for any NLP project?**

*Double-click this cell to write your answer:*

**1. Most affected text type:**
**Ans: The most affected text type is social media. This text type suffers the highest reduction percentage duirng preprocessing because it relies heavily on informal features. For example emojis, hashtags, mentions and non-standard capitalization/punctuation are either stripped or heavily normalized by standard cleaning pipelines.

**2. Least affected text type:**
**Ans: The least affected text type are news and nature of text. News is where the formal news articles generally experience the lowest percentage of token reduction during standard preprocessing. Nature of text is where news text strictly adheres to grammatical structure, proper syntax, and standard vocabulary. It lacks informal noises like emojis or hashtags. In other words preprocessing primarily removes common stop words while preserving the bulk of the original vocabulary.

**3. For customer review analysis:**
**Ans: For customer review analysis it would be the chosen approach where the minimal/lightweight preprocessing such as lowercasing, negation preservation, emoji/rating normalization. This will avoid aggressive stop-word removal or stemming. The reason for this is because reviewing data relies on context and subtle sentiment cues. Removing stop words strips crucial negations such as turning "not good" into "good". Also removing punctuation or emojis discards high-signal sentiment indicators. Lighweight processing retains exact semantics while keeping the text compatible with sentiment models.

**4. Main trade-offs to consider:**
**Ans: Main trade-offs to consider are Information Retention vs Noise Reduction, Stemming vs Lemmatization and pipeline speed vs model accuracy. First Information Retention vs Noise Reduction is removing stop words, punctuation, and casing shrinks the vocabulary size and speed training, but risks removing emotional nuance, negation amd key context. Second, Stemming vs Lemmatization is where stemming is fast but coarse such as producing non-words, whereas lemmatization preserves valid dictionary root forms at the cost of higher computational runtime. Lastly, pipeline speed vs model accuracy is where the complex preprocessing pipelines add execution latency during inference, whereas minimal preprocessing relies on heavier downstrram models such as transformers to phrase context.

---

## 🎯 Lab Summary and Reflection

Congratulations! You've completed a comprehensive exploration of NLP preprocessing techniques.

### 🔑 Key Concepts You've Mastered:

1. **Text Preprocessing Fundamentals** - Understanding why preprocessing is crucial
2. **Tokenization Techniques** - NLTK vs spaCy approaches and their trade-offs
3. **Stop Word Management** - When to remove them and when to keep them
4. **Morphological Processing** - Stemming vs lemmatization for different use cases
5. **Text Cleaning Strategies** - Basic vs advanced cleaning for different text types
6. **Pipeline Design** - Building modular, configurable preprocessing systems

### 🎓 Real-World Applications:
These techniques form the foundation for search engines, chatbots, sentiment analysis, document classification, machine translation, and information extraction systems.

### 💡 Key Insights to Remember:
- **No Universal Solution**: Different NLP tasks require different preprocessing approaches
- **Trade-offs Are Everywhere**: Balance information preservation with noise reduction
- **Context Matters**: The same technique can help or hurt depending on your use case
- **Experimentation Is Key**: Always test and measure impact on your specific task

---

**Excellent work completing Lab 02!** 🎉

For your reflection journal, focus on the insights you gained about when and why to use different techniques, the challenges you encountered, and connections you made to real-world applications.